<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/exercises/ex1/ex1_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML4NLP1
## Starting Point for Exercise 1, part II

This notebook is supposed to serve as a starting point and/or inspiration when starting exercise 1, part II.

One of the goals of this exercise is to make you acquainted with **skorch**. You will probably need to consult the [documentation](https://skorch.readthedocs.io/en/stable/).

## Use of generative AI

AI tools may be used in this exercise to help you explain code, debug, improve documentation, and explore alternatives. Any use must be documented in the *Declaration on the use of AI* at the end of this notebook (which tools were used and for which purposes). You remain responsible for understanding, checking, and testing everything you submit, and you must be able to explain any submitted text, code, analyses, and AI-assisted contributions. See the [CL guidelines on academic integrity and AI use](https://www.cl.uzh.ch/en/Studies/Services/academic-integrity.html).

# Setup

In [1]:
# Pin the library versions used in this course (the same ones as in Part I and the W04 tutorial).
# PyTorch is preinstalled on Colab; for a local installation see https://pytorch.org/get-started/locally/
%pip install -q scikit-learn==1.9.1 skorch==1.4.0 gdown

Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch
from torch import nn
import torch.nn.functional as F
from skorch import NeuralNetClassifier

import pandas as pd
import numpy as np
import csv
import re
import string
from collections import defaultdict

# Set seed for reproducibility
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

## Training a classifier and making predictions

In [ ]:
# Download dataset without hashtag splits
!gdown 1QP6YuwdKFNUPpvhOaAcvv2Pcp4JMbIRs
!gdown 1QVo7PZAdiZKzifK8kwhEr_umosiDCUx6
!gdown 1QbBeKcmG2ZyAEFB3AKGTgSWQ1YEMn2jl
!gdown 1QaZj6bI7_78ymnN8IpSk4gVvg-C9fA6X

In [ ]:
# Download dataset
!gdown 1QP6YuwdKFNUPpvhOaAcvv2Pcp4JMbIRs # x_train
!gdown 1QVo7PZAdiZKzifK8kwhEr_umosiDCUx6 # x_test
!gdown 1QbBeKcmG2ZyAEFB3AKGTgSWQ1YEMn2jl # y_train
!gdown 1QaZj6bI7_78ymnN8IpSk4gVvg-C9fA6X # y_test

In [7]:
with open('x_train.txt') as f:
    x_train = f.read().splitlines()
with open('y_train.txt') as f:
    y_train = f.read().splitlines()
with open('x_test.txt') as f:
    x_test = f.read().splitlines()
with open('y_test.txt') as f:
    y_test = f.read().splitlines()

UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 613: character maps to <undefined>

In [8]:
with open('x_train.txt', encoding='utf-8') as f:
    x_train = f.read().splitlines()

with open('y_train.txt', encoding='utf-8') as f:
    y_train = f.read().splitlines()

with open('x_test.txt', encoding='utf-8') as f:
    x_test = f.read().splitlines()

with open('y_test.txt', encoding='utf-8') as f:
    y_test = f.read().splitlines()

In [9]:
# Combine x_train and y_train into one dataframe
train_df = pd.DataFrame({'text': x_train, 'label': y_train})
# Combine x_test and y_test into one dataframe
test_df = pd.DataFrame({'text': x_test, 'label': y_test})
# Inspect the first 5 items in the train split
train_df.head()

,text,label
0,Klement Gottwaldi surnukeha palsameeriti ning ...,est
1,"Sebes, Joseph; Pereira Thomas (1961) (på eng)....",swe
2,भारतीय स्वातन्त्र्य आन्दोलन राष्ट्रीय एवम क्षे...,mai
3,"Après lo cort periòde d'establiment a Basilèa,...",oci
4,ถนนเจริญกรุง (อักษรโรมัน: Thanon Charoen Krung...,tha


### Data preparation

Prepare your dataset for this experiment using the same method as you did in part 1.

Get a subset of the train/test data that includes 20 languages. Include English (`eng`), German (`deu`), Dutch (`nld`), Danish (`dan`), Swedish (`swe`), Norwegian (either Bokmål `nob` or Nynorsk `nno`), and Japanese (`jpn`), plus 13 additional languages of your choice based on the items in the list of labels.

Unlike `sklearn`, the neural network cannot work with string labels: `skorch` and `CrossEntropyLoss` expect the labels as integer class indices `0, ..., n_classes - 1`. We therefore encode them with a [`LabelEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html).


In [10]:
# TODO: Create your train/test subsets of languages
# Note, make sure these are the same as what you used in Part 1!

selected_labels = sorted(['eng', 'deu', 'nld', 'dan', 'swe', 'nob', 'jpn'])
additional_labels = sorted(['fra', 'spa', 'ita', 'rus', 'ara', 'hin', 'zho', 'asm', 'heb', 'kan', 'kor', 'tha', 'vie'])
all_labels = sorted(selected_labels + additional_labels)

from sklearn.model_selection import train_test_split
# shuffle the train_df and test_df to ensure randomness and use 80% of the data for training and 20% for testing
train_test_df = pd.concat([train_df, test_df]).sample(frac=1, random_state=42).reset_index(drop=True)
train_test_df = train_test_df[train_test_df['label'].isin(all_labels)].drop_duplicates(subset=['text'])
x_train, x_test, y_train, y_test = train_test_split(
    train_test_df['text'], train_test_df['label'],
    test_size=0.2, stratify=train_test_df['label'], random_state=42)

In [ ]:
# TODO: With the following code, we wanted to ENCODE the labels, however, our cat was walking on the keyboard and some of it got changed. Can you fix it?
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder().fit(y_train)
# y_train, y_test = label_encoder.transform(y_train), label_encoder.transform(y_test) #.fit() returned a label encoder only, we need transform instead to avoid data leakage
y_train = label_encoder.transform(y_train)
print(label_encoder.classes_)
print(y_train)
print(y_test)

['ara' 'asm' 'dan' 'deu' 'eng' 'fra' 'heb' 'hin' 'ita' 'jpn' 'kan' 'kor'
 'nld' 'nob' 'rus' 'spa' 'swe' 'tha' 'vie' 'zho']
[2 2 5 ... 6 9 2]
[19 12 11 ...  8 18 14]


### Feature Extraction

In [12]:
# First, we extract some simple features as input for the neural network
# Note: x_train and y_train must refer to your 20-language subset from above, not to the full dataset
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(analyzer='char', ngram_range=(2, 2), max_features=100, binary=True)
X = vectorizer.fit_transform(x_train)

In [13]:
# We need to change the datatype to make it play nice with pytorch
X = X.astype(np.float32)
y = y_train.astype(np.int64)

In the following, we define a vanilla neural network with two hidden layers. The output layer should have as many outputs as there are classes. In addition, it should have a nonlinearity function.

In [17]:
# TODO: In the following, you can find a small (almost) working example of a neural network.
# Unfortunately, again, the cat messed up some of the code. Please fix the code such that it is executable. (Hint: the input and output sizes look a bit weird...)

class ClassifierModule(nn.Module):
    def __init__(
        self,
        input_size=100,
        num_units=200,
        hidden2=50, #for tuning the second hidden layer
        num_classes=20,
        nonlin=F.relu,
        dropout = 0.0
    ):
        super().__init__()
        #self.num_units = num_units
        self.nonlin = nonlin

        self.dense0 = nn.Linear(input_size, num_units)
        self.dense1 = nn.Linear(num_units, hidden2)
    
        self.dropout = nn.Dropout(dropout)

        self.output = nn.Linear(hidden2, num_classes)

    def forward(self, X, **kwargs):
        X = self.dropout(self.nonlin(self.dense0(X)))
        X = self.dropout(self.nonlin(self.dense1(X)))
        return self.output(X)


In [18]:
# Initialise the neural net classifier.
net = NeuralNetClassifier(
    ClassifierModule(
        input_size=X.shape[1],
        num_units=200,
        num_classes=len(label_encoder.classes_),
        nonlin=F.relu,
    ),
    max_epochs=20,
    criterion=nn.CrossEntropyLoss(),
    lr=0.1,
    device='cuda' if torch.cuda.is_available() else 'cpu',   # train on the GPU if Colab gives you one
)

In [19]:
# Train the classifier
net.fit(X, y)

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        2.6889       0.2840        2.4282  1.5000
      2        2.1142       0.3782        1.8362  1.4287
      3        1.6836       0.4470        1.5540  1.3745
      4        1.4483       0.5327        1.3560  1.4161
      5        1.2763       0.5721        1.2239  1.3397
      6        1.1739       0.5815        1.1539  1.0268
      7        1.1180       0.5852        1.1182  1.1042
      8        1.0845       0.5896        1.0974  1.0722
      9        1.0614       0.5918        1.0826  1.0731
     10        1.0433       0.5931        1.0729  1.0602
     11        1.0285       0.5956        1.0650  1.0487
     12        1.0159       0.5981        1.0592  1.0117
     13        1.0051       0.5990        1.0549  0.9877
     14        0.9955       0.6021        1.0511  1.0672
     15        0.9868       0.6040        1.0480  1.0392
     16        0.9789       0.6

,module,"ClassifierMod..., bias=True) )"
,criterion,CrossEntropyLoss()
,optimizer,<class 'torch.optim.sgd.SGD'>
,lr,0.1
,max_epochs,20
,batch_size,128
,iterator_train,<class 'torch...r.DataLoader'>
,iterator_valid,<class 'torch...r.DataLoader'>
,dataset,<class 'skorc...aset.Dataset'>
,callbacks,None
,predict_nonlinearity,'auto'


Note, you can also use `GridSearchCV` with `skorch` (it follows the `sklearn` API), but be aware that training a neural network takes much more time; a handful of hand-picked configurations, compared on the validation split that `skorch` holds out for you, is enough here.

Play around with 5 different sets of hyperparameters. For example, consider some of the following:

- layer sizes
- activation functions
- regularizers
- early stopping
- vectorizer parameters (`max_features`, `ngram_range`)

Report your best hyperparameter combination.

📝❓ What is the effect of your modifications on validation performance? Discuss potential reasons.

☝ Note, during model development, if you run into the infamous CUDA out-of-memory (OOM) error, try clearing the GPU memory either with `torch.cuda.empty_cache()` or restarting the runtime.


---

## Lab report

📝❓ Write your lab report here, addressing all questions in the notebook. For convenience, the questions are collected below; please keep your answers clearly marked.

### Questions

- 📝❓ Report your best hyperparameter combination.
- 📝❓ What is the effect of your modifications on validation performance? Discuss potential reasons.

## Declarations

Please fill in both declarations below by editing this cell (double-click it). Mark the applicable option by replacing ☐ with ☒ (copy the symbol from here: ☒) and complete the text field where required. Your reviewers will check that both declarations are present and completed.

### Contribution declaration

By submitting this work, the group declares that the contributions of the group members were:

☐ approximately equal

☐ not approximately equal

If the contributions were not approximately equal, no individual group member needs to be identified.

### Declaration on the use of AI

The group declares that any use of AI-based tools in preparing this work complied with the rules specified for this exercise.

☐ No AI-based tools were used.

☐ AI-based tools were used in accordance with the rules of the exercise.

If AI-based tools were used, briefly state the tool(s) used and the purpose(s) for which they were used:

